# P1 — Load & hygiene check

Loads the raw M5 CSVs with explicit dtypes, runs the structural/content assertion suite from
`docs/plan/P1-hygiene.md`, and writes `data/processed/sales_long.parquet` for downstream phases.

**Invariant:** no computation over `d_1914`-`d_1941` beyond the structural assertions below —
hygiene checks are scoped to the training range only.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("..") / "data"
PROCESSED_DIR = DATA_DIR / "processed"

N_DAYS = 1941  # d_1 .. d_1941, sales_train_evaluation.csv
DAY_COLS = [f"d_{i}" for i in range(1, N_DAYS + 1)]
ID_COLS = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

# Training range only (invariant 1): d_1914-d_1941 is the holdout. DAY_COLS above is for
# structural checks that may span the full file (shape, dtypes, column existence); any check
# that summarizes sales *values* must use TRAIN_DAY_COLS instead.
TRAIN_END = 1913
TRAIN_DAY_COLS = [f"d_{i}" for i in range(1, TRAIN_END + 1)]

## Load: sales_train_evaluation.csv

Load once with pandas' default dtype inference to measure the naive memory cost, then reload with
explicit dtypes (`int16` day columns, `category` id columns) to show the reduction the plan calls
for.

In [2]:
# Naive load: default dtype inference, to quantify the cost we're avoiding.
_naive = pd.read_csv(DATA_DIR / "sales_train_evaluation.csv")
naive_mb = _naive.memory_usage(deep=True).sum() / 1e6
print(f"naive dtype inference: {naive_mb:,.1f} MB")
del _naive

naive dtype inference: 485.8 MB


In [3]:
# Typed load: explicit dtypes per docs/plan/P1-hygiene.md.
sales_eval_dtypes = {**{c: "int16" for c in DAY_COLS}, **{c: "category" for c in ID_COLS}}
sales_eval = pd.read_csv(DATA_DIR / "sales_train_evaluation.csv", dtype=sales_eval_dtypes)

typed_mb = sales_eval.memory_usage(deep=True).sum() / 1e6
print(f"typed load: {typed_mb:,.1f} MB  (naive was {naive_mb:,.1f} MB, {naive_mb / typed_mb:,.1f}x larger)")
print(f"shape: {sales_eval.shape}")
sales_eval.dtypes.value_counts()

typed load: 122.6 MB  (naive was 485.8 MB, 4.0x larger)
shape: (30490, 1947)


int16       1941
category       1
category       1
category       1
category       1
category       1
category       1
Name: count, dtype: int64

The naive load is larger because day columns default to `int64` (8 bytes) instead of `int16`
(2 bytes), and id columns default to `object` (a separate string + pointer per row) instead of
`category` (each unique value stored once). Explicit dtypes keep the ~46-47M row long panel and
per-store feature partitions within the 15.9 GB RAM budget (invariant 3).

In [4]:
sales_eval.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0


## Load: remaining CSVs

Same typed-load discipline for `sales_train_validation.csv` (kept only to check agreement with the
evaluation file on `d_1`-`d_1913`, then dropped), `calendar.csv`, `sell_prices.csv`, and
`sample_submission.csv`.

In [5]:
# sales_train_validation.csv only carries d_1..d_1913 (validation-horizon file).
VALIDATION_DAY_COLS = [f"d_{i}" for i in range(1, 1913 + 1)]
sales_val_dtypes = {**{c: "int16" for c in VALIDATION_DAY_COLS}, **{c: "category" for c in ID_COLS}}
sales_val = pd.read_csv(DATA_DIR / "sales_train_validation.csv", dtype=sales_val_dtypes)

print(f"sales_train_validation: {sales_val.memory_usage(deep=True).sum() / 1e6:,.1f} MB, shape {sales_val.shape}")

sales_train_validation: 120.9 MB, shape (30490, 1919)


In [6]:
sales_val.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


In [7]:
calendar_dtypes = {
    "wm_yr_wk": "int16",
    "weekday": "category",
    "wday": "int8",
    "month": "int8",
    "year": "int16",
    "d": "category",
    "event_name_1": "category",
    "event_type_1": "category",
    "event_name_2": "category",
    "event_type_2": "category",
    "snap_CA": "int8",
    "snap_TX": "int8",
    "snap_WI": "int8",
}
calendar = pd.read_csv(DATA_DIR / "calendar.csv", dtype=calendar_dtypes, parse_dates=["date"])
print(f"calendar: {calendar.memory_usage(deep=True).sum() / 1e6:,.1f} MB, shape {calendar.shape}")

sell_prices_dtypes = {"store_id": "category", "item_id": "category", "wm_yr_wk": "int16", "sell_price": "float32"}
sell_prices = pd.read_csv(DATA_DIR / "sell_prices.csv", dtype=sell_prices_dtypes)
print(f"sell_prices: {sell_prices.memory_usage(deep=True).sum() / 1e6:,.1f} MB, shape {sell_prices.shape}")

sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
print(f"sample_submission: shape {sample_submission.shape}")

calendar: 0.2 MB, shape (1969, 14)


sell_prices: 61.8 MB, shape (6841121, 4)


sample_submission: shape (60980, 29)


In [8]:
calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [9]:
sell_prices.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


## Structural assertions

Each of these should fail if the data isn't shaped the way every later phase assumes.

### Unique id hygiene

In [10]:
# 30,490 unique id; id decomposes exactly into item_id + "_" + store_id + "_evaluation".
assert len(sales_eval) == 30_490, f"expected 30,490 rows, got {len(sales_eval)}"
assert sales_eval["id"].nunique() == 30_490, f"expected 30,490 unique ids, got {sales_eval['id'].nunique()}"

reconstructed_id = sales_eval["item_id"].astype(str) + "_" + sales_eval["store_id"].astype(str) + "_evaluation"
assert (sales_eval["id"].astype(str) == reconstructed_id).all(), "id does not decompose into item_id_store_id_evaluation for all rows"

print("id uniqueness and decomposition: OK")

id uniqueness and decomposition: OK


### sales_train eval vs. validation data mapping

In [11]:
# sales_train_evaluation.csv and sales_train_validation.csv agree on d_1..d_1913.
key_cols = ["item_id", "store_id"]
eval_sorted = sales_eval.sort_values(key_cols).reset_index(drop=True)
val_sorted = sales_val.sort_values(key_cols).reset_index(drop=True)

assert (eval_sorted[key_cols].astype(str).values == val_sorted[key_cols].astype(str).values).all(), \
    "evaluation and validation files do not carry the same (item_id, store_id) rows"

mismatches = (eval_sorted[VALIDATION_DAY_COLS].values != val_sorted[VALIDATION_DAY_COLS].values).sum()
assert mismatches == 0, f"{mismatches} day-cell mismatches between evaluation and validation files on d_1-d_1913"

print("evaluation vs validation agreement on d_1-d_1913: OK")

evaluation vs validation agreement on d_1-d_1913: OK


### calendar date hygiene check

In [12]:
# calendar.csv: 1,969 rows, contiguous dates 2011-01-29 -> 2016-06-19, no gaps.
assert len(calendar) == 1_969, f"expected 1,969 calendar rows, got {len(calendar)}"
assert calendar["date"].min() == pd.Timestamp("2011-01-29"), f"unexpected start date {calendar['date'].min()}"
assert calendar["date"].max() == pd.Timestamp("2016-06-19"), f"unexpected end date {calendar['date'].max()}"
assert calendar["date"].is_monotonic_increasing, "calendar dates are not sorted ascending"

gaps = calendar["date"].diff().dropna()
assert (gaps == pd.Timedelta(days=1)).all(), f"calendar has {int((gaps != pd.Timedelta(days=1)).sum())} gap(s) in daily sequence"

print("calendar row count and contiguity: OK")

calendar row count and contiguity: OK


### calendar week hygiene check

In [13]:
# wm_yr_wk maps consistently to date: each week value's dates form one contiguous <=7-day run.
wk_sizes = calendar.groupby("wm_yr_wk", observed=True).size()
assert wk_sizes.between(1, 7).all(), f"found wm_yr_wk group(s) outside 1-7 days: {wk_sizes[~wk_sizes.between(1, 7)]}"

for wk, grp in calendar.groupby("wm_yr_wk", observed=True):
    wk_gaps = grp["date"].diff().dropna()
    assert (wk_gaps == pd.Timedelta(days=1)).all(), f"wm_yr_wk={wk} dates are not contiguous"

# A <7-day group is only expected at the calendar's edges - the date range (2011-01-29 to
# 2016-06-19) was cut to those specific days for the competition, not to a wm_yr_wk boundary. A
# truncated week anywhere else would mean a real gap/mapping error, not an edge effect, so pin it
# down explicitly rather than relying on between(1, 7) alone.
first_wk, last_wk = calendar["wm_yr_wk"].iloc[0], calendar["wm_yr_wk"].iloc[-1]
truncated = wk_sizes[wk_sizes < 7]
unexpected_truncated = set(truncated.index) - {first_wk, last_wk}
assert not unexpected_truncated, f"truncated week(s) outside the calendar's start/end boundary: {unexpected_truncated}"

print(f"wm_yr_wk-to-date consistency: OK ({len(wk_sizes) - len(truncated)} full 7-day weeks, "
      f"{len(truncated)} truncated boundary week(s): {dict(truncated)})")

wm_yr_wk-to-date consistency: OK (281 full 7-day weeks, 1 truncated boundary week(s): {11621: np.int64(2)})


### store/department/category hygiene check

In [14]:
# 10 stores x 3,049 items; 3 states, 3 categories, 7 departments.
assert sales_eval["store_id"].nunique() == 10, f"expected 10 stores, got {sales_eval['store_id'].nunique()}"
assert sales_eval["item_id"].nunique() == 3_049, f"expected 3,049 items, got {sales_eval['item_id'].nunique()}"
assert sales_eval["state_id"].nunique() == 3, f"expected 3 states, got {sales_eval['state_id'].nunique()}"
assert sales_eval["cat_id"].nunique() == 3, f"expected 3 categories, got {sales_eval['cat_id'].nunique()}"
assert sales_eval["dept_id"].nunique() == 7, f"expected 7 departments, got {sales_eval['dept_id'].nunique()}"

# Full cross product: every item present in every store exactly once. This already follows from
# 30,490 unique ids (asserted above) decomposing 1:1 into (item_id, store_id) pairs, plus
# 3,049 x 10 == 30,490 - pigeonhole leaves no room for a missing item/store combo. Checked
# directly anyway so the invariant doesn't silently depend on that chain of reasoning.
combo_counts = sales_eval.groupby(["item_id", "store_id"], observed=True).size()
assert len(combo_counts) == 3_049 * 10, f"expected {3_049 * 10} (item_id, store_id) combos, got {len(combo_counts)}"
assert (combo_counts == 1).all(), "some (item_id, store_id) combo is missing or duplicated"

print("store/item/state/category/department cardinalities: OK")
print("every item present in every store exactly once: OK")

store/item/state/category/department cardinalities: OK
every item present in every store exactly once: OK


### sample_submission hygiene check

In [15]:
# sample_submission.csv: 60,980 rows in two 30,490-row id blocks.
assert len(sample_submission) == 60_980, f"expected 60,980 rows, got {len(sample_submission)}"

n_validation = sample_submission["id"].str.endswith("_validation").sum()
n_evaluation = sample_submission["id"].str.endswith("_evaluation").sum()
assert n_validation == 30_490, f"expected 30,490 _validation ids, got {n_validation}"
assert n_evaluation == 30_490, f"expected 30,490 _evaluation ids, got {n_evaluation}"

print("sample_submission id blocks: OK")

sample_submission id blocks: OK


## Content checks

Negative sales, price coverage / release dates, Christmas closures, sparsity, dead series, event
nulls, and duplicate price rows — all scoped to `d_1`-`d_1913` training range and the structural
`d_1914`-`d_1941` shape checks only, never any holdout summary statistic (invariant 1).

### Negative Sales check

In [16]:
# Negative or implausible sales - expect none. Scoped to d_1-d_1913 (invariant 1): this
# summarizes sales values (max, percentile), so it must not touch the d_1914-d_1941 holdout.
sales_matrix = sales_eval[TRAIN_DAY_COLS].values
n_negative = int((sales_matrix < 0).sum())
assert n_negative == 0, f"found {n_negative} negative sales values"

print(f"max daily sales value (d_1-d_{TRAIN_END}): {sales_matrix.max()}")
print(f"99.9th percentile daily sales value (d_1-d_{TRAIN_END}): {np.percentile(sales_matrix, 99.9):.1f}")
print("no negative sales: OK")

max daily sales value (d_1-d_1913): 763


99.9th percentile daily sales value (d_1-d_1913): 47.0
no negative sales: OK


### Price coverage check

In [17]:
# Price coverage: a missing sell_prices row means the item wasn't on sale that store-week.
# "Wasn't on sale" could mean pre-release (not yet in the catalog) or a mid-life gap (e.g. a
# stockout or delisting). sell_prices is a weekly catalog/price table, not an inventory table, so
# a stockout would show up as sales == 0 with the price row still present - not a missing row. The
# leading-gap-only story is verified below (not assumed): every (item_id, store_id) with any price
# data has one *contiguous* run of wm_yr_wk coverage from its release week through the final
# calendar week - no interior gaps, no early stop. If that weren't true, this cut would silently
# drop mid-life demand as if it were pre-release, so it's asserted rather than presumed.
#
# Local day-number Series (not mutated onto `calendar`, so no other cell can come to depend on
# this one having already run - see issue #11); used only to compute release_wk -> release_day
# below.
calendar_d_num = calendar["d"].astype(str).str.replace("d_", "", regex=False).astype("int16")
wk_to_first_day = calendar_d_num.groupby(calendar["wm_yr_wk"], observed=True).min()

release_info = (
    sell_prices.groupby(["item_id", "store_id"], observed=True)["wm_yr_wk"]
    .min()
    .rename("release_wk")
    .reset_index()
)
release_info["release_day"] = release_info["release_wk"].map(wk_to_first_day)

# Every (item_id, store_id) in sales_eval must have a price row somewhere (no untracked series).
missing_release = sales_eval[["item_id", "store_id"]].drop_duplicates().merge(
    release_info, on=["item_id", "store_id"], how="left"
)
assert missing_release["release_day"].notna().all(), \
    f"{missing_release['release_day'].isna().sum()} series have no sell_prices row at all"

# Contiguity + no-early-stop check: for each (item_id, store_id), its set of price wm_yr_wk values
# must span release_wk..last_calendar_wk with no interior gap.
all_wks = sorted(calendar["wm_yr_wk"].unique())
wk_position = {wk: i for i, wk in enumerate(all_wks)}
last_wk = all_wks[-1]

price_wk_sets = sell_prices.groupby(["item_id", "store_id"], observed=True)["wm_yr_wk"].apply(
    lambda s: tuple(sorted(s.unique()))
)
span = price_wk_sets.apply(lambda wks: wk_position[wks[-1]] - wk_position[wks[0]] + 1)
n_rows_in_span = price_wk_sets.apply(len)
interior_gap = price_wk_sets[span != n_rows_in_span]
trailing_gap = price_wk_sets[price_wk_sets.apply(lambda wks: wks[-1] != last_wk)]

assert interior_gap.empty, f"{len(interior_gap)} (item_id, store_id) combo(s) have an interior gap in price coverage"
assert trailing_gap.empty, f"{len(trailing_gap)} (item_id, store_id) combo(s) stop pricing before the final calendar week"

leading_gap_days = (release_info["release_day"] - 1).clip(lower=0, upper=N_DAYS)
pre_release_rows = int(leading_gap_days.sum())
pre_release_pct = pre_release_rows / (30_490 * N_DAYS) * 100

print("price coverage is a single contiguous run per series, ending at the final calendar week: OK")
print(f"pre-release rows: {pre_release_rows:,} ({pre_release_pct:.1f}% of the full melt)")

price coverage is a single contiguous run per series, ending at the final calendar week: OK
pre-release rows: 12,299,413 (20.8% of the full melt)


### Christmas closure check

In [18]:
# Christmas closures: every series should read zero on 25 Dec, 2011-2015 (all five fall within
# the training range, d_1913 or earlier). In practice a handful of FOODS items record small
# same-day sales despite the closure - a known minor M5 data quirk, not a loading bug. Report and
# bound it rather than hard-failing on a handful of cells out of 5 x 30,490 checked.
xmas_dates = pd.to_datetime([f"{y}-12-25" for y in range(2011, 2016)])
xmas_d_cols = calendar.loc[calendar["date"].isin(xmas_dates), "d"].astype(str).tolist()
assert len(xmas_d_cols) == 5, f"expected 5 Christmas dates in calendar, found {len(xmas_d_cols)}"

total_checked = 0
total_nonzero = 0
for d_col, xmas_date in zip(xmas_d_cols, xmas_dates):
    nonzero_mask = sales_eval[d_col] != 0
    nonzero = int(nonzero_mask.sum())
    total_checked += len(sales_eval)
    total_nonzero += nonzero
    if nonzero:
        print(f"{xmas_date.date()} ({d_col}): {nonzero} non-zero rows")
        print(sales_eval.loc[nonzero_mask, ["id", "cat_id", "store_id", d_col]].to_string(index=False))

nonzero_frac = total_nonzero / total_checked
print(f"\nChristmas non-zero cells: {total_nonzero} of {total_checked} ({nonzero_frac:.3%})")
assert nonzero_frac < 0.001, \
    f"{nonzero_frac:.3%} non-zero Christmas cells exceeds the 0.1% tolerance for the known data quirk"

2011-12-25 (d_331): 8 non-zero rows
                         id cat_id store_id  d_331
FOODS_3_586_CA_2_evaluation  FOODS     CA_2      6
FOODS_3_755_CA_2_evaluation  FOODS     CA_2      1
FOODS_3_406_CA_3_evaluation  FOODS     CA_3      1
FOODS_3_144_TX_3_evaluation  FOODS     TX_3      1
FOODS_3_226_TX_3_evaluation  FOODS     TX_3      1
FOODS_3_226_WI_2_evaluation  FOODS     WI_2      1
FOODS_3_238_WI_2_evaluation  FOODS     WI_2      1
FOODS_3_261_WI_2_evaluation  FOODS     WI_2      1
2012-12-25 (d_697): 11 non-zero rows
                         id cat_id store_id  d_697
FOODS_3_555_CA_2_evaluation  FOODS     CA_2      1
FOODS_3_586_CA_2_evaluation  FOODS     CA_2      1
FOODS_1_087_CA_3_evaluation  FOODS     CA_3      1
FOODS_3_151_CA_3_evaluation  FOODS     CA_3      1
FOODS_3_295_CA_3_evaluation  FOODS     CA_3      1
FOODS_3_694_CA_3_evaluation  FOODS     CA_3      1
FOODS_3_238_TX_3_evaluation  FOODS     TX_3      1
FOODS_3_377_TX_3_evaluation  FOODS     TX_3      1
FOODS_3_0

### Sparsity profile check

In [19]:
# Sparsity profile: zero fraction overall and by category/store/department. Expect ~68% overall.
# Scoped to d_1-d_1913 (invariant 1): a zero-fraction is a summary statistic over sales values,
# so it must not include the d_1914-d_1941 holdout.
row_zero_frac = (sales_eval[TRAIN_DAY_COLS] == 0).mean(axis=1)
overall_zero_frac = row_zero_frac.mean()
print(f"overall zero fraction (d_1-d_{TRAIN_END}): {overall_zero_frac:.1%} - expected ~68%")

for group_col in ["cat_id", "store_id", "dept_id"]:
    print(f"\nby {group_col}:")
    print(row_zero_frac.groupby(sales_eval[group_col], observed=True).mean().sort_values(ascending=False))

overall zero fraction (d_1-d_1913): 68.2% - expected ~68%

by cat_id:
cat_id
HOBBIES      0.772801
HOUSEHOLD    0.717696
FOODS        0.620212
dtype: float64

by store_id:
store_id
CA_4    0.721746
TX_1    0.708527
WI_2    0.707174
WI_3    0.701984
TX_3    0.699029
CA_2    0.691085
WI_1    0.689984
TX_2    0.664612
CA_1    0.639473
CA_3    0.596013
dtype: float64

by dept_id:
dept_id
HOBBIES_2      0.884784
HOUSEHOLD_2    0.807174
HOBBIES_1      0.732691
FOODS_2        0.680686
FOODS_1        0.631980
HOUSEHOLD_1    0.631077
FOODS_3        0.587879
dtype: float64


### Dead series check

In [20]:
# Dead series: items with no sales in the final 60 days. Count only - the handling decision
# (forecast zero vs let the model decide) is made in P7, not here.
# "final 60 days" is scoped to the training range (d_1854-d_1913, using the global TRAIN_END):
# using the file's actual last 60 columns would pull in the d_1914-d_1941 holdout, which
# invariant 1 forbids touching here.
last_60_train_cols = [f"d_{i}" for i in range(TRAIN_END - 59, TRAIN_END + 1)]
dead_mask = (sales_eval[last_60_train_cols] == 0).all(axis=1)
n_dead = int(dead_mask.sum())
print(f"dead series (zero sales in final 60 training days, d_{TRAIN_END - 59}-d_{TRAIN_END}): {n_dead} of 30,490")

dead series (zero sales in final 60 training days, d_1854-d_1913): 955 of 30,490


### Event distribution check

In [21]:
# Event columns: nulls are meaningful ("no event"), not missing data. Do not fill them.
event_cols = ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]
print(calendar[event_cols].isna().sum())
print(f"\n{len(calendar) - calendar['event_name_1'].isna().sum()} of {len(calendar)} days carry a primary event")
print("event nulls are left as-is (mean 'no event'): OK")

event_name_1    1807
event_type_1    1807
event_name_2    1964
event_type_2    1964
dtype: int64

162 of 1969 days carry a primary event
event nulls are left as-is (mean 'no event'): OK


### Duplicate price per store/item/week check

In [22]:
# Duplicate price rows: expect none on (store_id, item_id, wm_yr_wk).
n_dup = int(sell_prices.duplicated(subset=["store_id", "item_id", "wm_yr_wk"]).sum())
assert n_dup == 0, f"found {n_dup} duplicate (store_id, item_id, wm_yr_wk) rows in sell_prices"

print("no duplicate price rows: OK")

no duplicate price rows: OK


## Output: melt, join, drop pre-release rows, write parquet

Melt `sales_eval` to long `(id, d, sales)`, join `calendar` on `d` and `sell_prices` on
`(store_id, item_id, wm_yr_wk)`, drop rows with no price (pre-release), and persist
`data/processed/sales_long.parquet` per the data contract in `docs/plan/README.md`.

In [23]:
# Melt to long format.
sales_long = pd.melt(sales_eval, id_vars=ID_COLS, value_vars=DAY_COLS, var_name="d_label", value_name="sales")
sales_long["sales"] = sales_long["sales"].astype("int16")

FULL_MELT_ROWS = 30_490 * N_DAYS
assert len(sales_long) == FULL_MELT_ROWS == 59_181_090, \
    f"expected {FULL_MELT_ROWS:,} rows after melt, got {len(sales_long):,}"

print(f"melted: {len(sales_long):,} rows")

melted: 59,181,090 rows


In [24]:
# Join calendar on d, then sell_prices on (store_id, item_id, wm_yr_wk). Both are 1:1 joins
# (calendar is unique per d, sell_prices has no duplicate (store_id, item_id, wm_yr_wk) rows per
# the assertion above), so row count should not change.
sales_long = sales_long.merge(calendar, left_on="d_label", right_on="d", how="left")

# Numeric day column, computed locally from d_label - this section is self-contained and does not
# depend on a mutation made in an earlier, conceptually unrelated cell (issue #11).
sales_long["d"] = sales_long["d_label"].str.replace("d_", "", regex=False).astype("int16")
sales_long = sales_long.drop(columns=["d_label"])

sales_long = sales_long.merge(sell_prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")

assert len(sales_long) == FULL_MELT_ROWS, \
    f"row count changed across joins: {len(sales_long):,} vs {FULL_MELT_ROWS:,} expected (a join was not 1:1)"

print(f"after calendar + price joins: {len(sales_long):,} rows, {sales_long.shape[1]} columns")
print(f"missing sell_price (pre-release candidates): {sales_long['sell_price'].isna().sum():,}")

after calendar + price joins: 59,181,090 rows, 22 columns


missing sell_price (pre-release candidates): 12,299,413


In [ ]:
# Drop pre-release rows: a missing sell_price means the item wasn't on sale that store-week.
# Validate first: all pre-release rows (missing sell_price) should have sales == 0, confirming
# that the leading-gap cut removes only structural zeros, not actual demand.
prerelease_mask = sales_long["sell_price"].isna()
prerelease_sales = sales_long.loc[prerelease_mask, "sales"]
nonzero_prerelease = (prerelease_sales != 0).sum()
assert nonzero_prerelease == 0, \
    f"found {nonzero_prerelease} pre-release rows with non-zero sales (expected all to be zero)"

pre_drop_rows = len(sales_long)
sales_long = sales_long[sales_long["sell_price"].notna()].reset_index(drop=True)
post_drop_rows = len(sales_long)

dropped = pre_drop_rows - post_drop_rows
dropped_pct = dropped / pre_drop_rows * 100
print(f"pre-release rows all have zero sales: OK")
print(f"dropped {dropped:,} pre-release rows ({dropped_pct:.1f}% of the full melt)")
print(f"cross-check against the leading-gap estimate above: {pre_release_rows:,} ({pre_release_pct:.1f}%)")

assert 45_000_000 <= post_drop_rows <= 48_000_000, \
    f"post-cut row count {post_drop_rows:,} outside the expected ~46-47M range"

print(f"post-cut: {post_drop_rows:,} rows")

In [26]:
# Final schema check against the data contract in docs/plan/README.md:
# id category, d int16, sales int16, date datetime64, wm_yr_wk int16, calendar columns,
# item_id/dept_id/cat_id/store_id/state_id category, sell_price float32.
sales_long["sell_price"] = sales_long["sell_price"].astype("float32")

expected_dtype_kinds = {
    "id": "category", "item_id": "category", "dept_id": "category", "cat_id": "category",
    "store_id": "category", "state_id": "category",
    "d": "int16", "sales": "int16", "wm_yr_wk": "int16",
    "date": "datetime64[ns]", "sell_price": "float32",
}
for col, expected in expected_dtype_kinds.items():
    actual = str(sales_long[col].dtype)
    assert actual == expected, f"{col}: expected dtype {expected}, got {actual}"

print("schema check: OK")
print(f"memory usage: {sales_long.memory_usage(deep=True).sum() / 1e6:,.1f} MB")
sales_long.dtypes

schema check: OK
memory usage: 1,785.5 MB


id                    category
item_id               category
dept_id               category
cat_id                category
store_id              category
state_id              category
sales                    int16
date            datetime64[ns]
wm_yr_wk                 int16
weekday               category
wday                      int8
month                     int8
year                     int16
d                        int16
event_name_1          category
event_type_1          category
event_name_2          category
event_type_2          category
snap_CA                   int8
snap_TX                   int8
snap_WI                   int8
sell_price             float32
dtype: object

In [27]:
# Write parquet per the pinned path in docs/plan/README.md.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output_path = PROCESSED_DIR / "sales_long.parquet"
sales_long.to_parquet(output_path, index=False)

file_size_mb = output_path.stat().st_size / 1e6
print(f"wrote {output_path}: {post_drop_rows:,} rows, {file_size_mb:,.1f} MB")

wrote ..\data\processed\sales_long.parquet: 46,881,677 rows, 227.7 MB


## Summary of findings

**Structural assertions** — all passed: 30,490 unique `id` decomposing exactly into
`item_id_store_id_evaluation`; evaluation and validation files agree on `d_1`-`d_1913`;
`calendar.csv` has 1,969 contiguous daily rows (2011-01-29 → 2016-06-19); `wm_yr_wk` maps to
contiguous ≤7-day date runs, with the one truncated (2-day) week confirmed to fall only at the
calendar's final boundary (`wm_yr_wk=11621`), not mid-series; 10 stores × 3,049 items, 3 states,
3 categories, 7 departments, confirmed to form the full 30,490-row cross product (every item
present in every store exactly once); `sample_submission.csv` has the expected 60,980-row / two
30,490-block shape.

**Content checks** (all scoped to `d_1`-`d_1913`, the training range — invariant 1):
- No negative sales (max daily value 763, 99.9th percentile 47.0).
- **Price coverage / release dates:** every series has at least one `sell_prices` row; the
  leading (pre-release) gap accounts for 12,299,413 rows. Confirmed (not assumed) that a missing
  price row means pre-release rather than a mid-life gap (stockout, delisting): all 30,490
  `(item_id, store_id)` combos with price data show one contiguous `wm_yr_wk` run from release
  week through the final calendar week — zero interior gaps, zero combos stopping early.
- **Christmas closures:** 25 Dec is a near-universal closure, not an absolute one — 60 non-zero
  cells out of 152,450 checked (0.039%) across 2011-2015, all small counts (1-6 units) in
  `FOODS_3_*` items. Assertion relaxed to a 0.1% tolerance rather than hard-zero; see
  `docs/plan/DECISIONS.md`.
- **Sparsity:** 68.2% zero over `d_1`-`d_1913`, matching the plan's expectation (an external
  planning-time prior — see `docs/plan/DECISIONS.md`, pre-P0 section — now confirmed against the
  actual data). Highest in HOBBIES_2 (88.5%) and store CA_4 (72.2%); lowest in FOODS_3 (58.8%) and
  store CA_3 (59.6%).
- **Dead series:** 955 of 30,490 (3.1%) have zero sales in the final 60 *training* days
  (`d_1854`-`d_1913`). Handling deferred to P7.
- Event-column nulls (1,807-1,964 of 1,969 calendar rows, depending on column) left unfilled —
  they mean "no event."
- No duplicate `(store_id, item_id, wm_yr_wk)` price rows.

**Output:** full melt produced exactly 59,181,090 rows (30,490 × 1,941); dropping pre-release
rows (missing `sell_price`) removed 12,299,413 rows (20.8%), landing at **46,881,677 rows**.
This matches the plan's "~46-47M rows" target but not its "~12-13% dropped" prose — that
inconsistency is in the brief itself (`docs/plan/P1-hygiene.md`), not a defect here; the drop is
fully explained by the leading pre-release gap with zero trailing gap. Schema verified
column-by-column against the `docs/plan/README.md` data contract. Wrote
`data/processed/sales_long.parquet` (227.7 MB, 22 columns).

**Gate:** all checks pass. See `docs/plan/DECISIONS.md` for the recorded decisions, deviations,
and evidence.